In [14]:
import pandas as pd
import random, string, functions, os
from os.path import join
from pathlib import Path


def generate_key(length):
    """원본 길이에 맞는 임의 키 생성 (숫자만)"""
    return ''.join(random.choices(string.digits, k=length))

def anonymize_resident_columns(df):
    """
    DataFrame에서 '주민'을 포함하는 모든 컬럼을 찾아서
    각 값들을 동일 길이의 임의 키로 변환하고,
    기존 컬럼을 덮어씌웁니다.
    """
    mapping = {}

    # '주민'이 포함된 컬럼만 선택
    target_columns = [col for col in df.columns if "주민" in col]

    for col in target_columns:
        new_values = []
        for original in df[col]:
            if original not in mapping:
                # 하이픈 제거 후 길이 확인
                clean = original.replace("-", "")
                key = generate_key(len(clean))
                # 하이픈 위치 유지
                if "-" in original:
                    parts = original.split("-")
                    key = key[:len(parts[0])] + "-" + key[len(parts[0]):]
                mapping[original] = key
            new_values.append(mapping[original])
        
        # 기존 컬럼 덮어쓰기
        df[col] = new_values

    return df

In [15]:
path_to_read = r"C:\Users\DATA\Desktop\개발용 나우리 데이터"
path_to_write = r"C:\Users\DATA\Desktop\개발용 나우리 데이터\output"
if not os.path.exists(path_to_write):
    os.mkdir(path_to_write)

In [16]:
filenames = [f.name for f in Path(path_to_read).iterdir() if f.is_file()]
filenames

for fn in filenames:
    if fn.endswith(".xlsx"):
        df = pd.read_excel(join(path_to_read, fn))
    elif fn.endswith(".parquet"):
        df = pd.read_parquet(join(path_to_read, fn))
    else :
        print(f"{fn}은 지원하지 않는 형식입니다. 해당 확장자를 코드에 반영하세요.")
        break
    
    df_anonymized = anonymize_resident_columns(df)
    df_anonymized.to_parquet(join(path_to_write, f"anonymized_{fn}"), index=False)
    print(f"{fn} 처리 완료.")

감면조회새창_20260601_1127.parquet 처리 완료.
개인회생새창_20260601_1724.parquet 처리 완료.
계좌조회새창_20260601_1126.parquet 처리 완료.
법조치조회새창_20260601_1151.parquet 처리 완료.
보증인새창_20260601_1127.parquet 처리 완료.
약정분납새창_20260601_1355.parquet 처리 완료.
입금조회새창_20260601_1125_당월.parquet 처리 완료.
채무자조회새창_20260601_1123.parquet 처리 완료.
